# B3 · nn.Module

手写 `w - lr * w.grad` 只能对付一两个参数。真实模型有几十万个参数，需要一套机制把参数管起来。

那就是 `nn.Module`。

**学完这一节你应该能**
- 定义一个自己的 Module，知道 `forward` 和 `__call__` 的关系
- 用 `parameters()` 拿到所有参数、数清楚有多少
- 用 `state_dict()` 存盘和加载
- 知道 `train()` / `eval()` 改变了什么

In [ ]:
import torch


def check(name, ok, hint=""):
    if ok:
        print(f"✓ {name}")
    else:
        print(f"✗ {name}")
        if hint:
            print(f"   提示：{hint}")

## 1. 第一个 Module

Module 只做三件事：**登记参数**、**定义计算**、**提供存取接口**。

In [ ]:
import torch.nn as nn


class Tiny(nn.Module):
    def __init__(self):
        super().__init__()                 # 必须先调，否则参数登记不上
        self.fc = nn.Linear(3, 1)          # 这一行自动把这个层登记进来

    def forward(self, x):
        return self.fc(x)


model = Tiny()
print(model)
print("\n参数数量：", sum(p.numel() for p in model.parameters()), "（3 个权重 + 1 个偏置）")

### 调用时为什么写 `model(x)` 而不是 `model.forward(x)`

因为 `nn.Module.__call__` 在 `forward` 外面包了一层钩子（hook），做前向预处理、反向钩子等。直接调 `forward` 会绕过这些机制。

**结论：永远写 `model(x)`。**

In [ ]:
x = torch.randn(2, 3)
print("model(x) 输出 shape:", tuple(model(x).shape))
print("两个写法数值一样：", torch.allclose(model(x), model.forward(x)))
print("但 model(x) 会触发 hook，forward(x) 不会。平时用 model(x)。")

## 2. 参数：`parameters()` 和 `named_parameters()`

`named_parameters()` 会告诉你每个参数叫什么、长什么样，排查问题时最常用。

In [ ]:
for name, p in model.named_parameters():
    print(f"{name:<12} shape={tuple(p.shape)!s:<10} 需要梯度={p.requires_grad}")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n总计 {total} 个参数")

In [ ]:
# 稍微大一点的模型，参数都在哪些层
class TwoLayer(nn.Module):
    def __init__(self, d_in=8, d_hidden=16, d_out=4):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


m = TwoLayer()
for name, p in m.named_parameters():
    print(f"{name:<10} {tuple(p.shape)}")

n = sum(p.numel() for p in m.parameters())
print(f"\n总参数 {n} = 8x16+16 (fc1) + 16x4+4 (fc2)")
print("再核对一遍：", 8 * 16 + 16 + 16 * 4 + 4)

## 3. 存盘与加载：`state_dict()`

`state_dict` 就是一个「参数名 → 张量」的字典。存它，不存整个模型对象。

In [ ]:
import io

m = TwoLayer()
sd = m.state_dict()
print("state_dict 的键：", list(sd.keys()))
print("每项的 shape  ：", {k: tuple(v.shape) for k, v in sd.items()})

# 存到一个内存缓冲区（真实场景用 torch.save(sd, "xxx.pt")）
buf = io.BytesIO()
torch.save(sd, buf)
buf.seek(0)
loaded = torch.load(buf, weights_only=True)

m2 = TwoLayer()                          # 结构必须一模一样
m2.load_state_dict(loaded)

x = torch.randn(3, 8)
print("\n加载后输出是否完全一致：", torch.allclose(m(x), m2(x)))

**注意**：`state_dict` 只存参数，不存结构。所以加载时必须先 `TwoLayer()` 建好同样的结构。

想连结构一起存，可以用 `torch.save(model, path)`，但那样依赖类定义，跨项目容易出问题。**实践中一律存 state_dict。**

## 4. `train()` 和 `eval()`

这两个方法本身不改参数，只改一个布尔开关 `self.training`。某些层会读这个开关：

| 层 | train 模式 | eval 模式 |
|---|---|---|
| `Dropout` | 随机丢弃一部分 | 全部保留 |
| `BatchNorm` | 用当前 batch 的统计量 | 用累计的滑动平均 |

**推理前忘了 `eval()`，是新手最常见的隐蔽 bug** —— 结果每次都不一样，还以为模型没训好。

In [ ]:
torch.manual_seed(0)

drop = nn.Dropout(p=0.5)
x = torch.ones(10)

drop.train()
print("train 模式两次输出：")
print("  ", drop(x))
print("  ", drop(x))

drop.eval()
print("eval 模式两次输出：")
print("  ", drop(x))
print("  ", drop(x))

In [ ]:
class WithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(4, 4)
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        return self.drop(self.fc(x))


m = WithDropout()
x = torch.randn(2, 4)

print("默认（train）：", m.training)
print("两次输出差多少：", (m(x) - m(x)).abs().max().item())

m.eval()
print("\neval 之后     ：", m.training)
print("两次输出差多少：", (m(x) - m(x)).abs().max().item())

## 5. `Sequential` 和 `ModuleList`

层多了以后，一个个写 `self.fc1 / self.fc2 / ...` 很啰嗦。

In [ ]:
seq = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4),
)
print(seq)
print("输出 shape:", tuple(seq(torch.randn(2, 8)).shape))

# 带名字的写法，方便按名字取
named = nn.Sequential()
named.add_module("fc1", nn.Linear(8, 16))
named.add_module("act", nn.ReLU())
named.add_module("fc2", nn.Linear(16, 4))
print("\n按名字取：", named.fc2)

In [ ]:
# ModuleList：只是「装层的列表」，不定义前向，用起来灵活
class Stack(nn.Module):
    def __init__(self, n_layers=3, d=8):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(d, d) for _ in range(n_layers)])

    def forward(self, x):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return x


s = Stack()
print("层数：", len(s.layers), " 参数：", sum(p.numel() for p in s.parameters()))
print("输出：", tuple(s(torch.randn(2, 8)).shape))

## 6. 练习

In [ ]:
# 练习 1：写一个只有一层的 Module，输入 4 维、输出 2 维
class OneLayer(nn.Module):
    def __init__(self):
        super().__init__()
        # ← 在这里定义一层

    def forward(self, x):
        return x                     # ← 改这一行


m = OneLayer()
check("练习 1：输出 shape 应为 (3, 2)，参数应为 10 个",
      tuple(m(torch.randn(3, 4)).shape) == (3, 2)
      and sum(p.numel() for p in m.parameters()) == 10,
      hint="self.fc = nn.Linear(4, 2)，forward 里 return self.fc(x)")

In [ ]:
# 练习 2：存盘再加载，要求输出完全一致
m = TwoLayer()
x = torch.randn(2, 8)
out1 = m(x)

buf = io.BytesIO()
torch.save(m.state_dict(), buf)
buf.seek(0)

m2 = TwoLayer()
# ← 在这里加载参数

check("练习 2：加载后输出一致",
      torch.allclose(out1, m2(x)),
      hint="m2.load_state_dict(torch.load(buf, weights_only=True))")

In [ ]:
# 练习 3：让 dropout 在推理时不起作用，两次输出应该完全一样
torch.manual_seed(0)
m = WithDropout()
x = torch.randn(2, 4)

# ← 在这里切换到正确模式

diff = (m(x) - m(x)).abs().max().item()
check("练习 3：两次输出应完全一样", diff == 0.0,
      hint="model.eval()")

In [ ]:
# 练习 4：冻结 fc1 的参数（不参与训练）
m = TwoLayer()
# ← 在这里冻结 m.fc1

check("练习 4：可训练参数应只剩 fc2 的 68 个",
      sum(p.numel() for p in m.parameters() if p.requires_grad) == 68,
      hint="for p in m.fc1.parameters(): p.requires_grad = False")

In [ ]:
# 练习 5：用 nn.Sequential 重写 TwoLayer 的结构（8 -> 16 -> 4）
seq = None                      # ← 改这一行

check("练习 5：输出 shape 应为 (2, 4) 且参数 212 个",
      seq is not None
      and tuple(seq(torch.randn(2, 8)).shape) == (2, 4)
      and sum(p.numel() for p in seq.parameters()) == 212,
      hint="nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,4))")

## 收尾

回答这三个：

1. `model(x)` 和 `model.forward(x)` 有什么区别？为什么推荐前者？
2. `state_dict` 里存的是什么？为什么不直接存整个模型对象？
3. 推理前忘了 `eval()` 会发生什么？哪些层受影响？

下一节：**B4 · 训练循环**。